# 06. 시계열과 LSTM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2038prob/defense-ai-textbook-code/blob/main/notebooks/%EC%8B%A4%EC%8A%B5_06_%EC%8B%9C%EA%B3%84%EC%97%B4%EA%B3%BC_LSTM.ipynb)

「국방 인공지능」 실습 코드 — 이 장의 파이썬 코드를 순서대로 모았습니다. **맨 위 setup 셀부터 위에서 아래로** 실행하세요(앞 셀에서 정의한 변수를 뒤 셀이 이어 씁니다). ⚠️ 표시가 붙은 셀은 개념 설명용 조각으로, 그대로는 실행되지 않을 수 있습니다.

전체 저장소: https://github.com/2038prob/defense-ai-textbook-code


In [ ]:
# === 실습 환경 준비 (Colab/로컬 공통) — 가장 먼저 실행하세요 ===
# 필요한 패키지를 설치하고(이미 있으면 건너뜀) 공통 유틸을 정의합니다.
import importlib.util, subprocess, sys
for pkg in ["torch", "torchvision", "numpy", "matplotlib", "scipy"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import random, numpy as np, torch

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)

## BPTT: 시간축으로 펼친 신경망의 역전파

### 🔬 실습 — 절단 BPTT에서 은닉 상태를 분리하기

> ⚠️ **설명용 조각** — 아래는 절단 BPTT의 개념을 보여 주는 코드로, `rnn`·`optimizer`·`x_chunks` 등은 별도로 정의해야 합니다(그대로는 실행되지 않습니다).

In [ ]:
# x_chunks: 길이 K로 나눈 입력 청크들의 리스트
h = torch.zeros(num_layers, batch_size, hidden_dim, device=device)
for x_chunk, y_chunk in x_chunks:
    optimizer.zero_grad()
    out, h = rnn(x_chunk, h)
    loss = criterion(head(out), y_chunk)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
    optimizer.step()

    # 값은 다음 청크로 전달하되, 이전 계산 그래프와는 분리
    h = h.detach()

## 실습: LSTM으로 시계열 예측

### 🔬 실습 — 슬라이딩 윈도우 + LSTM 예측기 학습

① **슬라이딩 윈도우 데이터셋**: 과거 \(T\)시점으로 다음 1시점을 예측.


In [ ]:
import torch
from torch.utils.data import Dataset

class WindowDataset(Dataset):
    def __init__(self, series, T=40, H=1):   # series: (L, d)
        self.series, self.T, self.H = series, T, H
    def __len__(self):
        return self.series.shape[0] - self.T - self.H + 1
    def __getitem__(self, idx):
        x = self.series[idx : idx+self.T]                    # (T, d)
        y = self.series[idx+self.T : idx+self.T+self.H, 0]   # (H,) 첫 채널
        return x, y

② **모델 정의**: LSTM의 마지막 시점 은닉 상태를 선형층으로 예측에 매핑.


In [ ]:
import torch.nn as nn

class LSTMForecaster(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, horizon=1, dropout=0.0):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True,           # (B, T, d) 입력
                            dropout=dropout if num_layers > 1 else 0.0)
        self.fc = nn.Linear(hidden_dim, horizon)
    def forward(self, x):
        out, _ = self.lstm(x)            # (B, T, hidden)
        return self.fc(out[:, -1, :])    # 마지막 시점 -> (B, horizon)

③ **정규화(훈련 통계로만)**: 데이터 누수 방지.


In [ ]:
class Normalizer:
    def fit(self, data):                 # data: (L, d)
        self.mean = data.mean(0); self.std = data.std(0).clamp(min=1e-8)
    def transform(self, data):
        return (data - self.mean) / self.std

④ **학습 루프**: 합성 데이터(sin+추세+잡음)로 MSE 예측 학습.


In [ ]:
# --- (실습용 합성 데이터) sin + 추세 + 잡음의 2채널 시계열로 train_loader 구성 ---
import numpy as np
import torch
from torch.utils.data import DataLoader

L = 1000
t = np.arange(L)
ch0 = np.sin(2 * np.pi * t / 50) + 0.002 * t + 0.1 * np.random.randn(L)
ch1 = np.cos(2 * np.pi * t / 40) + 0.001 * t + 0.1 * np.random.randn(L)
series = torch.tensor(np.stack([ch0, ch1], axis=1), dtype=torch.float32)  # (L, 2)

# 훈련 구간 통계로만 정규화(데이터 누수 방지)
n_train = int(L * 0.8)
norm = Normalizer(); norm.fit(series[:n_train])
series_n = norm.transform(series)

train_ds = WindowDataset(series_n[:n_train], T=40, H=1)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
print("윈도우 수:", len(train_ds))

In [ ]:
import torch.optim as optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = LSTMForecaster(2, 64, num_layers=2, horizon=1, dropout=0.2).to(device)
criterion, optimizer = nn.MSELoss(), optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(1, 21):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = criterion(model(xb), yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()

## 푸리에 변환에서 log-Mel 스펙트로그램까지

### 🔬 실습 — 파형에서 log-Mel 특징 만들기

> ⚠️ **설명용 조각** — 아래는 개념을 보여 주는 코드로, 그대로는 실행되지 않을 수 있습니다. (외부 데이터·모델 파일이 필요합니다)

In [ ]:
import torch
import torchaudio

waveform, sr = torchaudio.load("comm_audio.wav")  # (channel, samples)
waveform = waveform.mean(dim=0, keepdim=True)     # mono
if sr != 16000:
    waveform = torchaudio.functional.resample(waveform, sr, 16000)
    sr = 16000

mel = torchaudio.transforms.MelSpectrogram(
    sample_rate=sr,
    n_fft=400,          # 25 ms at 16 kHz
    win_length=400,
    hop_length=160,     # 10 ms
    n_mels=80,
    power=2.0,
)(waveform)
log_mel = torch.log(mel.clamp_min(1e-10))
print(waveform.shape, log_mel.shape)

## 정렬 문제와 CTC의 blank

### 🔬 실습 — 간단한 CTC greedy decoder 구현


In [ ]:
def ctc_collapse(tokens, blank="_"):
    collapsed = []
    prev = None
    for tok in tokens:
        if tok != prev:          # 먼저 연속 중복 압축
            collapsed.append(tok)
        prev = tok
    return [tok for tok in collapsed if tok != blank]

path = ["ㅌ", "ㅏ", "ㅏ", "_", "ㅏ", "ㅇ", "ㅇ", "_"]
print(ctc_collapse(path))  # ['ㅌ', 'ㅏ', 'ㅏ', 'ㅇ']

## 전술 프롬프팅과 contextual biasing

### 🔬 실습 — 프롬프트, 무음 확률, 저신뢰 구간을 함께 기록하기

> ⚠️ **설명용 조각** — 아래는 개념을 보여 주는 코드로, 그대로는 실행되지 않을 수 있습니다. (외부 데이터·모델 파일이 필요합니다)

In [ ]:
import whisper

model = whisper.load_model("base")
result = model.transcribe(
    "comm_audio.wav",
    language="ko",
    task="transcribe",
    temperature=0.0,
    condition_on_previous_text=False,
    initial_prompt=(
        "보고 형식: 호출부호, 시각, 지점, 행동. "
        "용어 후보: KILO-21, ALPHA 고지, 집결지 BRAVO"
    )
)

review = []
for seg in result["segments"]:
    avg_logprob = float(seg.get("avg_logprob", -99.0))
    no_speech = float(seg.get("no_speech_prob", 1.0))
    needs_review = (avg_logprob < -1.0 or no_speech > 0.6)
    review.append({
        "start": seg["start"],
        "end": seg["end"],
        "text": seg["text"],
        "avg_logprob": avg_logprob,
        "no_speech_prob": no_speech,
        "needs_review": needs_review,
    })

for item in review:
    if item["needs_review"]:
        print("확인 필요:", item)

## Whisper 베이스라인과 도메인 용어 편향

> ⚠️ **설명용 조각** — 아래는 개념을 보여 주는 코드로, 그대로는 실행되지 않을 수 있습니다. (외부 데이터·모델 파일이 필요합니다)

In [ ]:
import whisper

model = whisper.load_model("base")
result = model.transcribe(
    "comm_audio.wav",
    language="ko",
    temperature=0.0,
    condition_on_previous_text=False,
    initial_prompt="용어집: 호출부호, 부대명, 장비명, 표준 숫자 읽기"
)
print(result["text"])
for seg in result["segments"][:3]:
    print(seg["start"], seg["end"], seg.get("avg_logprob"),
          seg.get("no_speech_prob"))

## 평가: WER 하나로는 부족하다

### 🔬 실습 — WER와 핵심어 Recall을 함께 계산하기


In [ ]:
def edit_distance(ref, hyp):
    dp = list(range(len(hyp) + 1))
    for i, r in enumerate(ref, 1):
        new = [i]
        for j, h in enumerate(hyp, 1):
            new.append(min(new[-1] + 1,       # insertion
                           dp[j] + 1,          # deletion
                           dp[j-1] + (r != h)))# substitution
        dp = new
    return dp[-1]

def wer(reference, hypothesis):
    r, h = reference.split(), hypothesis.split()
    return edit_distance(r, h) / max(1, len(r))

def keyword_recall(reference, hypothesis, keywords):
    ref_set = {k for k in keywords if k in reference}
    hit = sum(k in hypothesis for k in ref_set)
    return hit / max(1, len(ref_set))

ref = "호출부호 A12 14시 지점으로 이동"
hyp = "호출부호 A20 14시 지점 이동"
keys = ["A12", "14시"]
print("WER:", wer(ref, hyp))
print("핵심어 Recall:", keyword_recall(ref, hyp, keys))

---

실습 코드 저장소: https://github.com/2038prob/defense-ai-textbook-code  ·  교재 「국방 인공지능」
